# Практика 15. Критерії узгодженості й таблиці спряженості: критерій хі-квадрат

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса  
**Середньорічна температура:** 11.5 °C  
**Сезонна амплітуда:** 13 °C

У цій практиці генеруємо власний кліматичний набір за чотири роки, досліджуємо зв'язок між сезоном і категорією температурного відхилення, а також перевіряємо розподіли категорій критерієм хі-квадрат.

## Побудова кліматичного набору

Для кожного місяця 2021–2024 років температура коливається навколо середньорічної за косинусною сезонною моделлю з випадковим шумом. Категорія відхилення визначається відносно базової температури: нижче −3 °C — «холодніше», вище +3 °C — «тепліше», інакше — «звичайно».

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chisquare, chi2_contingency

np.random.seed(42)
base_temp = 11.5
amplitude = 13
city = "Одеса"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        difference = temp - base_temp
        if difference < -3:
            norm_category = "холодніше"
        elif difference > 3:
            norm_category = "тепліше"
        else:
            norm_category = "звичайно"
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_category,
        })

climate = pd.DataFrame(rows)
print(f"Місто: {city}; рядків у наборі: {len(climate)}")
display(climate.head(12))

## Завдання 1. Таблиця спряженості власного варіанта

Побудуємо таблиці частот і часток відхилень усередині кожного сезону. У нормованій таблиці сума значень кожного рядка дорівнює 1, тож частки можна порівнювати між сезонами незалежно від кількості спостережень.

In [ ]:
season_order = ["зима", "весна", "літо", "осінь"]
category_order = ["холодніше", "звичайно", "тепліше"]
table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
table = table.reindex(index=season_order, columns=category_order, fill_value=0)
table_normalized = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"],
    normalize="index",
).reindex(index=season_order, columns=category_order, fill_value=0)

print("Спостережені частоти:")
display(table)
print("Частки в межах кожного сезону:")
display(table_normalized.round(3))

most_common_pair = table.stack().idxmax()
print(
    f"Найчастіша комбінація: сезон «{most_common_pair[0]}» і категорія "
    f"«{most_common_pair[1]}» ({table.loc[most_common_pair]} спостережень)."
)

У цій таблиці сезони мають однакову кількість місяців за кожен рік, а температурна категорія створена з сезонного коливання. Тому очікувано, що тепліші відхилення частіше спостерігаються влітку, а холодніші — взимку. Нормування `normalize="index"` ділить кожну клітинку на суму свого рядка: кожен рядок стає розподілом часток, що сумуються до 1, і сезонні профілі легше порівнювати за пропорціями, а не лише за кількістю.

## Завдання 2. Критерій незалежності хі-квадрат

H₀: сезон і категорія відхилення від норми незалежні. H₁: між ними є зв'язок. Порівняємо фактичні частоти з очікуваними за незалежності; найбільша абсолютна різниця показує клітинку з найбільшим розходженням.

In [ ]:
alpha = 0.05
chi2_stat, p_independence, dof, expected = chi2_contingency(table)
expected_table = pd.DataFrame(expected, index=table.index, columns=table.columns)
difference_table = (table - expected_table).abs()
largest_difference_cell = difference_table.stack().idxmax()
reject_independence = p_independence < alpha

print(f"χ² = {chi2_stat:.4f}, ступені вільності = {dof}, p-value = {p_independence:.6g}")
print("Очікувані частоти за H₀:")
display(expected_table.round(2))
print("Абсолютні різниці між observed і expected:")
display(difference_table.round(2))
print(
    f"Найбільша розбіжність: «{largest_difference_cell[0]}» × "
    f"«{largest_difference_cell[1]}» (observed={table.loc[largest_difference_cell]}, "
    f"expected={expected_table.loc[largest_difference_cell]:.2f})."
)
if reject_independence:
    print(f"За α={alpha:.2f} відхиляємо H₀: дані свідчать про зв'язок між сезоном і температурною категорією.")
else:
    print(f"За α={alpha:.2f} немає достатніх підстав відхилити H₀; це не доводить незалежність змінних.")

Очікувану частоту клітинки за незалежності обчислюють як `сума рядка × сума стовпця / загальна сума`. Це еквівалентно припущенню, що загальна частка кожної категорії відхилення однакова всередині кожного сезону. Велике p-value означало б лише недостатньо доказів проти незалежності, а не доказ того, що змінні незалежні.

## Завдання 3. Перевірка очікуваних частот і за потреби об'єднання категорій

Перевіримо всі клітинки матриці expected за практичним правилом `E ≥ 5`. Якщо є клітинки нижче порога, об'єднаємо «холодніше» і «звичайно» в категорію «не тепліше» та повторимо тест. Це зберігає зміст порівняння з категорією «тепліше», водночас збільшуючи очікувані частоти.

In [ ]:
low_expected_mask = expected_table < 5
print("Клітинки з expected < 5:")
if low_expected_mask.to_numpy().any():
    display(expected_table.where(low_expected_mask).stack().rename("Очікувана частота").to_frame())
    print("Умова E ≥ 5 не виконується для всіх клітинок; об'єднуємо «холодніше» і «звичайно».")
    climate["відхилення_об'єднане"] = climate["відхилення_від_норми"].replace(
        {"холодніше": "не тепліше", "звичайно": "не тепліше"}
    )
    merged_order = ["не тепліше", "тепліше"]
    merged_table = pd.crosstab(climate["сезон"], climate["відхилення_об'єднане"])
    merged_table = merged_table.reindex(index=season_order, columns=merged_order, fill_value=0)
    merged_chi2, merged_p, merged_dof, merged_expected = chi2_contingency(merged_table)
    merged_expected_table = pd.DataFrame(merged_expected, index=merged_table.index, columns=merged_table.columns)
    print("Об'єднана таблиця частот:")
    display(merged_table)
    print("Очікувані частоти після об'єднання:")
    display(merged_expected_table.round(2))
    print(f"Після об'єднання всі expected ≥ 5: {(merged_expected_table >= 5).to_numpy().all()}")
    print(f"Повторний тест: χ²={merged_chi2:.4f}, dof={merged_dof}, p-value={merged_p:.6g}")
    print(
        "Висновок після об'єднання: "
        + ("відхиляємо незалежність за α=0.05." if merged_p < alpha else "немає достатніх підстав відхилити незалежність за α=0.05.")
    )
else:
    merged_table = table.copy()
    merged_expected_table = expected_table.copy()
    merged_p = p_independence
    print("Усі очікувані частоти не менші за 5; об'єднувати категорії не потрібно.")

Якщо навіть після об'єднання категорій expected залишаються замалими, наближення хі-квадрат може бути ненадійним. У такому разі треба об'єднати категорії змістовно (або, для таблиці 2×2, розглянути точний критерій Фішера), а не трактувати асимптотичне p-value як точне.

## Завдання 4. Критерій узгодженості для сезонів

Кожен із чотирьох сезонів містить по три місяці на рік, отже за чотири роки очікуємо по 12 спостережень на сезон. Перевіримо H₀ про рівномірний розподіл сезонних міток.

In [ ]:
season_counts = climate["сезон"].value_counts().reindex(season_order, fill_value=0)
expected_season_counts = np.full(len(season_order), len(climate) / len(season_order))
season_fit = chisquare(f_obs=season_counts.to_numpy(), f_exp=expected_season_counts)

print("Спостережені частоти сезонів:")
display(season_counts.rename("Спостережено").to_frame())
print("Очікувані частоти:", expected_season_counts)
print(f"χ²={season_fit.statistic:.4f}, p-value={season_fit.pvalue:.4f}")
if season_fit.pvalue < alpha:
    print("Відхиляємо H₀: розподіл сезонних міток відрізняється від рівномірного.")
else:
    print("Не відхиляємо H₀: немає підстав вважати розподіл сезонних міток відмінним від рівномірного.")

Для повного набору з чотирьох років кожен сезон має бути представлений однакову кількість разів за побудовою даних. Тому узгодженість із пропорцією 25% / 25% / 25% / 25% очікувана: тест перевіряє структуру календарних міток, а не випадковий розподіл температур.

## Завдання 5. Власна заявлена пропорція температурних категорій

**H₀:** частки категорій «холодніше», «звичайно» і «тепліше» дорівнюють відповідно 40%, 20% і 40%. Це обґрунтоване наближення для побудованої косинусної моделі: більшість місяців мають виражене сезонне відхилення за межами ±3 °C, а перехідні місяці частіше потрапляють у категорію «звичайно». Перевіримо пропорцію критерієм узгодженості.

In [ ]:
category_counts = climate["відхилення_від_норми"].value_counts().reindex(category_order, fill_value=0)
hypothesized_proportions = np.array([0.40, 0.20, 0.40])
expected_category_counts = hypothesized_proportions * len(climate)
category_fit = chisquare(f_obs=category_counts.to_numpy(), f_exp=expected_category_counts)

print("Спостережені та очікувані частоти:")
display(pd.DataFrame({
    "Спостережено": category_counts,
    "Очікується за H₀": expected_category_counts,
}, index=category_order))
print(f"χ²={category_fit.statistic:.4f}, p-value={category_fit.pvalue:.6f}")
if category_fit.pvalue < alpha:
    print("За α=0.05 відхиляємо H₀: дані статистично значуще не узгоджуються з пропорцією 40% / 20% / 40%.")
else:
    print("За α=0.05 немає достатніх підстав відхилити H₀ про пропорцію 40% / 20% / 40%; це не доводить, що пропорція точно правильна.")

## Контрольні питання

1. **Чому в статистиці χ² квадрат відхилення і ділення на очікувану частоту?** Квадрат усуває взаємне скорочення додатних і від'ємних відхилень та надає більшу вагу великим розбіжностям. Ділення на expected масштабує відхилення відносно типового розміру категорії: однакова абсолютна різниця важливіша для рідкісної категорії, ніж для дуже численної.

2. **Як обчислюють expected у таблиці спряженості?** Для кожної клітинки `expected = сума рядка × сума стовпця / загальна сума`. За незалежності частка категорії стовпця має бути однаковою для кожного рядка, тому очікувана кількість дорівнює розміру рядка, помноженому на загальну частку стовпця.

3. **Що означає велике p-value?** Це означає, що даних недостатньо, щоб відхилити H₀ на обраному рівні значущості. Воно не доводить незалежності чи точності заявленої пропорції.

4. **Що робити, якщо expected < 5?** Спершу вказати конкретні проблемні клітинки, потім змістовно об'єднати рідкісні категорії й повторити тест; для таблиці 2×2 із малими частотами можна застосувати точний критерій Фішера. Малі expected роблять асимптотичне наближення розподілом χ² ненадійним, тому p-value може бути неточним.

## Підсумок

На власному наборі згенерованих температур Одеси побудовано таблицю спряженості сезону й категорії відхилення від норми, перевірено незалежність та умову expected ≥ 5, перевірено рівномірність сезонів і окрему заявлену пропорцію температурних категорій.